# Dataset tests

Run one 32-token prompt from a text dataset through a single model using the analyzer pipeline. If needed, swap `DatasetPromptSource` for `RandomTokenPromptSource` without changing the rest of the code.

In [14]:
import pandas as pd


df = pd.read_parquet('Qwen3_4B_512tok.parquet')

In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRELIMINARY EDA — Attention Features | Qwen2.5-0.5B-Instruct
# ═══════════════════════════════════════════════════════════════════════════════
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

# ── 0. LOAD ────────────────────────────────────────────────────────────────────
sources = df['prompt_source'].unique()
SRC_COLOR = {s: c for s, c in zip(sources, ['#01696f', '#d19900'])}

DYNAMIC_FEATS = [
    'effective_rank_Q', 'effective_rank_K', 'effective_rank_A',
    'r95_Q', 'r95_K', 'r95_A',
    'q_sim_consecutive', 'k_sim_consecutive',
    'attention_entropy', 'attention_gini',
    'attention_row_var_weighted', 'look_back',
    'diagonal_mass_1', 'diagonal_mass_5',
    'diagonal_mass_1_shifted_1',
    'sink_mass_token_0', 'sink_mass_max',
    'WqWk_svd_alignment',
]

# ── 1. HEATMAP layer × head per feature chiave ────────────────────────────────
lh = df.groupby(['layer_idx', 'head_idx'])[DYNAMIC_FEATS].mean().reset_index()

for feat in ['attention_entropy', 'diagonal_mass_1', 'sink_mass_token_0',
             'effective_rank_Q', 'effective_rank_A', 'look_back',
             'q_sim_consecutive', 'WqWk_svd_alignment']:
    pivot = lh.pivot(index='head_idx', columns='layer_idx', values=feat)
    fig = px.imshow(
        pivot,
        color_continuous_scale='RdBu_r' if 'align' in feat else 'Viridis',
        title=f'<b>{feat}</b> — layer × head heatmap',
        labels=dict(x='Layer', y='Head', color=feat)
    )
    fig.update_xaxes(title_text='Layer')
    fig.update_yaxes(title_text='Head')
    fig.show()

# ── 2. BOXPLOT per source (key dynamic feats) ─────────────────────────────────
for feat in ['attention_entropy', 'diagonal_mass_1', 'sink_mass_token_0',
             'effective_rank_Q', 'effective_rank_K', 'look_back',
             'q_sim_consecutive', 'attention_gini']:
    fig = go.Figure()
    for src in sources:
        fig.add_trace(go.Box(
            y=df[df['prompt_source'] == src][feat],
            name=src.split('_')[0],
            marker_color=SRC_COLOR[src],
            boxmean='sd'
        ))
    fig.update_layout(
        title=f'<b>{feat}</b> — random_vocab vs wikitext',
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5)
    )
    fig.update_yaxes(title_text=feat)
    fig.show()

# ── 3. LAYER PROFILE: mean ± std ribbon ───────────────────────────────────────
ls = df.groupby(['layer_idx', 'prompt_source'])[DYNAMIC_FEATS].agg(['mean', 'std']).reset_index()
ls.columns = ['_'.join(c).strip('_') for c in ls.columns]

for feat in ['attention_entropy', 'diagonal_mass_1', 'sink_mass_token_0',
             'effective_rank_Q', 'look_back', 'WqWk_svd_alignment']:
    fig = go.Figure()
    for src in sources:
        sub = ls[ls['prompt_source'] == src].sort_values('layer_idx')
        x = sub['layer_idx'].tolist()
        m = sub[f'{feat}_mean']
        s = sub[f'{feat}_std']
        fig.add_trace(go.Scatter(
            x=x, y=m, mode='lines+markers',
            name=src.split('_')[0],
            line=dict(color=SRC_COLOR[src], width=2)
        ))
        fig.add_trace(go.Scatter(
            x=x + x[::-1],
            y=pd.concat([m + s, (m - s).iloc[::-1]]).tolist(),
            fill='toself', fillcolor=SRC_COLOR[src],
            opacity=0.15, line=dict(width=0),
            showlegend=False, hoverinfo='skip'
        ))
    fig.update_layout(
        title=f'<b>{feat}</b> across layers — mean ± std',
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5)
    )
    fig.update_xaxes(title_text='Layer')
    fig.update_yaxes(title_text=feat)
    fig.show()

# ── 4. CORRELATION MATRIX (dynamic feats) ────────────────────────────────────
corr = df[DYNAMIC_FEATS].corr()
fig = px.imshow(corr, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='<b>Correlation matrix</b> — dynamic attention features')
fig.update_xaxes(title_text='Feature', tickangle=-45)
fig.update_yaxes(title_text='Feature')
fig.show()

# ── 5. SCATTER: sink_mass_token_0 vs attention_entropy ────────────────────────
fig = px.scatter(
    df, x='attention_entropy', y='sink_mass_token_0',
    color='prompt_source', color_discrete_map=SRC_COLOR,
    opacity=0.25, trendline='lowess',
    title='<b>Attention sink vs entropy</b> — each point = one head × prompt',
    labels={'attention_entropy': 'Attention entropy',
            'sink_mass_token_0': 'Sink mass token 0'}
)
fig.show()

# ── 6. RANK-1 HEADS: heads dove eff_rank_Q ≤ 2 ────────────────────────────────
rank1 = df[df['effective_rank_Q'] <= 2]
pivot_r1 = (rank1.groupby(['layer_idx', 'head_idx'])
                  .size().reset_index(name='count')
                  .pivot(index='head_idx', columns='layer_idx', values='count')
                  .fillna(0))
fig = px.imshow(pivot_r1, color_continuous_scale='Reds',
                title='<b>Rank-1 Q head frequency</b> — eff_rank_Q ≤ 2 (count across prompts)',
                labels=dict(x='Layer', y='Head', color='count'))
fig.update_xaxes(title_text='Layer')
fig.update_yaxes(title_text='Head')
fig.show()

# ── 7. QUICK STATS ─────────────────────────────────────────────────────────────
print("=== RANK-1 / SINK / DIAGONAL SUMMARY ===")
print(f"Rank-1 Q heads (eff_rank_Q ≤ 2): {len(rank1)}/{len(df)} = {len(rank1)/len(df)*100:.1f}%")
print(f"High sink tok0 (> 0.8):           {(df['sink_mass_token_0'] > 0.8).sum()} rows")
print(f"High diagonal (diag_1 > 0.8):     {(df['diagonal_mass_1']   > 0.8).sum()} rows")

print("\n=== ENTROPY by source ===")
print(df.groupby('prompt_source')['attention_entropy'].describe())

print("\n=== eff_rank_Q by source ===")
print(df.groupby('prompt_source')['effective_rank_Q'].describe())

print("\n=== look_back by source ===")
print(df.groupby('prompt_source')['look_back'].describe())

print("\n=== Top correlates with attention_entropy ===")
print(corr['attention_entropy'].abs().sort_values(ascending=False).head(10))

print("\n=== Top correlates with diagonal_mass_1 ===")
print(corr['diagonal_mass_1'].abs().sort_values(ascending=False).head(10))

print("\n=== Top correlates with sink_mass_token_0 ===")
print(corr['sink_mass_token_0'].abs().sort_values(ascending=False).head(10))

=== RANK-1 / SINK / DIAGONAL SUMMARY ===
Rank-1 Q heads (eff_rank_Q ≤ 2): 0/23040 = 0.0%
High sink tok0 (> 0.8):           3892 rows
High diagonal (diag_1 > 0.8):     214 rows

=== ENTROPY by source ===
                                      count      mean       std       min  \
prompt_source                                                               
random_vocab                        11520.0  2.178799  1.178879  0.001302   
wikitext_wikitext-103-raw-v1_train  11520.0  1.934667  0.953483  0.001574   

                                         25%       50%       75%       max  
prompt_source                                                               
random_vocab                        1.233042  2.101061  3.082950  4.940494  
wikitext_wikitext-103-raw-v1_train  1.160041  1.880119  2.592405  5.034296  

=== eff_rank_Q by source ===
                                      count       mean        std        min  \
prompt_source                                                         